# Segmentación de clientes mediante aprendizaje no supervisado

## Objetivo
Aplicar técnicas de clustering para identificar grupos homogéneos de clientes a partir de variables demográficas y de consumo.

## Entregables esperados
- Análisis exploratorio del dataset
- Preprocesamiento y selección de variables
- Entrenamiento con al menos dos algoritmos de clustering
- Comparación de resultados
- Conclusiones y posibles aplicaciones de negocio


## 1. Configuración inicial

Ejecuta esta celda al inicio para preparar librerías, rutas y estilo de gráficos.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

BASE_DIR = Path.cwd()
FIGURES_DIR = BASE_DIR / "figuras"
FIGURES_DIR.mkdir(exist_ok=True)

DATASET_PATH = BASE_DIR / "Mall_Customers.csv"
DATASET_PATH


## 2. Carga del dataset

Si tu archivo tiene otro nombre, modifica `DATASET_PATH` en la celda anterior.


In [ ]:
df = pd.read_csv(DATASET_PATH)
df.head()


## 3. Exploración inicial del conjunto de datos

Completa esta sección con comentarios sobre dimensiones, tipos de datos, nulos y estadísticas descriptivas.


In [ ]:
print("Shape:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())
print("\nInfo:")
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.describe(include="all")


## 4. Análisis exploratorio visual

Genera y guarda los gráficos principales. Después interpreta cada uno en una celda Markdown.


In [ ]:
numeric_cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

df[numeric_cols].hist(figsize=(10, 6))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "histogramas_variables.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[numeric_cols])
plt.title("Boxplots de variables numéricas")
plt.savefig(FIGURES_DIR / "boxplots_variables.png", dpi=300)
plt.show()


In [ ]:
sns.pairplot(df[numeric_cols])
plt.savefig(FIGURES_DIR / "pairplot_variables.png", dpi=300)
plt.show()


## 5. Preprocesamiento

Justifica qué columnas usarás para clustering. La recomendación base es excluir `CustomerID` y trabajar con las variables numéricas principales.


In [ ]:
features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
X = df[features].copy()
X.head()


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:5]


## 6. Búsqueda del número óptimo de clusters para K-Means

Usa método del codo y silhouette score. Después, justifica el valor final elegido para `k`.


In [ ]:
k_values = range(2, 11)
inertia = []
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertia.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

metrics_kmeans = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertia,
    "silhouette": silhouette_scores
})
metrics_kmeans


In [ ]:
plt.plot(metrics_kmeans["k"], metrics_kmeans["inertia"], marker="o")
plt.title("Elbow Method")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.savefig(FIGURES_DIR / "elbow_method.png", dpi=300)
plt.show()


In [ ]:
plt.plot(metrics_kmeans["k"], metrics_kmeans["silhouette"], marker="o")
plt.title("Silhouette Score por número de clusters")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette Score")
plt.savefig(FIGURES_DIR / "silhouette_scores.png", dpi=300)
plt.show()


## 7. Entrenamiento del modelo K-Means final

Ajusta `best_k` según tus métricas y explicación del análisis previo.


In [ ]:
best_k = 5

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster_kmeans"] = kmeans.fit_predict(X_scaled)
df[["cluster_kmeans"] + features].head()


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="cluster_kmeans",
    palette="tab10"
)
plt.title("Clusters obtenidos con K-Means")
plt.savefig(FIGURES_DIR / "kmeans_clusters.png", dpi=300)
plt.show()


## 8. Segundo algoritmo de clustering

La plantilla base usa `AgglomerativeClustering`. Si prefieres `DBSCAN`, puedes sustituir esta sección y justificar el cambio.


In [ ]:
agg = AgglomerativeClustering(n_clusters=best_k)
df["cluster_agg"] = agg.fit_predict(X_scaled)

silhouette_kmeans = silhouette_score(X_scaled, df["cluster_kmeans"])
silhouette_agg = silhouette_score(X_scaled, df["cluster_agg"])

print("Silhouette K-Means:", silhouette_kmeans)
print("Silhouette Agglomerative:", silhouette_agg)


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="cluster_agg",
    palette="Set2"
)
plt.title("Clusters obtenidos con Agglomerative Clustering")
plt.savefig(FIGURES_DIR / "agglomerative_clusters.png", dpi=300)
plt.show()


## 9. Resumen e interpretación de clusters

Calcula promedios por grupo y redacta una interpretación de negocio para cada cluster.


In [ ]:
summary_kmeans = df.groupby("cluster_kmeans")[features].mean().round(2)
summary_kmeans["size"] = df["cluster_kmeans"].value_counts().sort_index()
summary_kmeans


In [ ]:
summary_agg = df.groupby("cluster_agg")[features].mean().round(2)
summary_agg["size"] = df["cluster_agg"].value_counts().sort_index()
summary_agg


## 10. Comparación de algoritmos

Resume ventajas, limitaciones y métrica principal de cada método.


In [ ]:
comparison = pd.DataFrame([
    {
        "Metodo": "K-Means",
        "Numero_clusters": df["cluster_kmeans"].nunique(),
        "Silhouette": silhouette_kmeans,
        "Observacion": "Simple y fácil de interpretar"
    },
    {
        "Metodo": "Agglomerative",
        "Numero_clusters": df["cluster_agg"].nunique(),
        "Silhouette": silhouette_agg,
        "Observacion": "Útil para comparación jerárquica"
    }
])
comparison


## 11. Conclusiones

Completa esta sección con:
- el algoritmo que funcionó mejor,
- interpretación general de los segmentos,
- aplicaciones de negocio,
- limitaciones del análisis,
- mejoras futuras.
